# AffectProject on Colab: DAiSEE download, continuity audit, feature extraction

This notebook only **launches** the project's scripts (`src/*.py`) with `!python ...`; all the logic lives in the repo.
Edit code on the Mac, `git push`, then rerun cell 2 here (it pulls).

| Session | Runtime | Run cells | Result on Drive |
|---|---|---|---|
| A | CPU is enough | 1 → 9 | `audit/`, `sample_clips/` |
| B | **T4 GPU** | 1 → 5, then 10 → 14 | `features/` |

`/content` (code, the DAiSEE download) is wiped when a runtime ends; only `MyDrive/AffectProject` persists.
Every long step resumes if rerun after a disconnect.

## 1. Settings

In [ ]:
REPO = "pauldsun/affect-detection"
DAISEE_URL = ""            # paste the dataset link you got from the DAiSEE form
DRIVE_OUT = "/content/drive/MyDrive/AffectProject"   # everything worth keeping goes here
KEEP_ARCHIVE_ON_DRIVE = False   # True: copy the download to Drive once so session B skips it (needs that much free Drive space)

CODE, DATA, ARCHIVE = "/content/affect", "/content/data", "/content/DAiSEE_download"

## 2. Drive + code
Private repo: make a read-only fine-grained GitHub token for this repo and save it in the 🔑 Secrets panel as `GH_TOKEN`
(toggle notebook access on). Public repo: nothing to do.

In [ ]:
import os, re, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT, exist_ok=True)

if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} pull --ff-only
else:
    url = f"https://github.com/{REPO}.git"
    try:
        from google.colab import userdata
        url = f"https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git"
    except Exception:
        print("no GH_TOKEN secret: cloning as a public repo")
    !git clone -q {url} {CODE}
%cd {CODE}
!git log -1 --format="code at %h: %s (%cr)"

## 3. Download DAiSEE
Optional first look at what the link points to: `content-type: application/zip` / `octet-stream` = a file (good);
`text/html` = a web page, so open the link in your browser and copy the real download URL from the browser's downloads list.

In [ ]:
!curl -sIL "{DAISEE_URL}" | grep -iE "^(HTTP|location|content-type|content-length|content-disposition)"

In [ ]:
DRIVE_ARCHIVE = f"{DRIVE_OUT}/DAiSEE_download"
if os.path.exists(ARCHIVE):
    print("already downloaded")
elif os.path.exists(DRIVE_ARCHIVE):
    print("copying the saved download from Drive ..."); shutil.copy(DRIVE_ARCHIVE, ARCHIVE)
else:
    assert DAISEE_URL, "paste the link into DAISEE_URL (cell 1) and rerun cell 1"
    if "drive.google.com" in DAISEE_URL:
        !pip -q install gdown
        !gdown --fuzzy "{DAISEE_URL}" -O {ARCHIVE}
    else:
        url = DAISEE_URL.replace("dl=0", "dl=1")          # Dropbox share links
        !wget -c -q --show-progress -O {ARCHIVE} "{url}"

with open(ARCHIVE, "rb") as f:
    magic = f.read(8)
KIND = "zip" if magic.startswith(b"PK") else "rar" if magic.startswith(b"Rar!") else "7z" if magic.startswith(b"7z") else None
print(f"{os.path.getsize(ARCHIVE) / 1e9:.2f} GB, type: {KIND}")
if KIND is None:
    !head -c 400 {ARCHIVE}; echo
    raise RuntimeError("not an archive (shown above): the link gave a web page or an error, not the dataset. Delete it: !rm " + ARCHIVE)
if KEEP_ARCHIVE_ON_DRIVE and not os.path.exists(DRIVE_ARCHIVE):
    print("copying to Drive ..."); shutil.copy(ARCHIVE, DRIVE_ARCHIVE)

## 4. Unpack and find the folders

In [ ]:
os.makedirs(DATA, exist_ok=True)
if not os.listdir(DATA):
    if KIND == "zip":
        !unzip -q {ARCHIVE} -d {DATA}
    elif KIND == "rar":
        !unrar x -inul {ARCHIVE} {DATA}/
    else:
        !7z x -bd -o{DATA} {ARCHIVE} > /dev/null

!find {DATA} -maxdepth 3 -not -name "*.avi" | head -25
datasets = glob.glob(f"{DATA}/**/DataSet", recursive=True)
labels = glob.glob(f"{DATA}/**/AllLabels.csv", recursive=True)
inner = [p for e in ("zip", "rar", "7z") for p in glob.glob(f"{DATA}/**/*.{e}", recursive=True)]
DATASET = datasets[0] if datasets else None
LABELS = labels[0] if labels else None
n_vid = len(glob.glob(f"{DATASET}/**/*.avi", recursive=True)) if DATASET else 0
print(f"\nDATASET = {DATASET}\nLABELS  = {LABELS}\nvideos found: {n_vid}")
if inner:
    print(f"{len(inner)} archives inside the archive, e.g. {inner[:3]}: these need unpacking too (send me this output)")
if not DATASET or not LABELS:
    print("layout differs from expected: look at the tree above and set DATASET / LABELS by hand in a new cell")

## 5. Is everything in place?
Quick check before running anything long.

In [ ]:
!python -c "import cv2, pandas, tqdm; print('cv2', cv2.__version__, '| pandas', pandas.__version__)"
!ffprobe -version | head -1
!df -h /content | tail -1

## 6. Continuity audit, quick look (10 subjects, a few minutes)
Decodes video only, no models, no GPU. Results are cached per subject in `audit/scan/`, so cell 7 reuses them.

In [ ]:
!python src/audit_continuity.py "{DATASET}" --labels "{LABELS}" --out {DRIVE_OUT}/audit --max-subjects 10

## 7. Continuity audit, all subjects (rerun after a disconnect: it resumes)

In [ ]:
!python src/audit_continuity.py "{DATASET}" --labels "{LABELS}" --out {DRIVE_OUT}/audit

## 8. Read the audit

In [ ]:
from IPython.display import Image, display
print(open(f"{DRIVE_OUT}/audit/summary.txt").read())
for m in sorted(glob.glob(f"{DRIVE_OUT}/audit/montage_*.png")):
    print(os.path.basename(m)); display(Image(m, width=800))

## 9. Copy a sample of raw clips to Drive (to watch on the Mac)

In [ ]:
!python src/sample_clips.py "{DATASET}" --labels "{LABELS}" --out {DRIVE_OUT}/sample_clips

---
# Session B: feature extraction (GPU)
Runtime → Change runtime type → **T4 GPU**, then run cells 1 → 5 and continue here.

## 10. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: Runtime > Change runtime type > T4 GPU"
!python -c "import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"

## 11. Install OpenFace 3.0 (Linux version of the README's macOS setup)
- `--no-deps`: the pip package pins old numpy/Pillow that would replace Colab's CUDA-ready stack; Colab already has
  everything else it imports, and the smoke test (cell 12) confirms it works.
- OpenFace repo pinned to the commit used on the Mac. Weights fetched directly (`openface download` skips them), sizes verified.
- Same patch as `scripts/patch_openface.sh` (the hardcoded `/work/...` path), done in Python because that script is macOS-only.

In [ ]:
OF = f"{CODE}/OpenFace-3.0"
if not os.path.isdir(OF):
    !git clone -q https://github.com/CMU-MultiComp-Lab/OpenFace-3.0.git {OF}
    !git -C {OF} checkout -q 662a555a8566ae3aec8139cb8c72acf6d06e0eb3
!pip -q install --no-deps openface-test==0.1.26
!pip -q install timm==1.0.15 tensorboardX

import urllib.request
BASE = "https://huggingface.co/nutPace/openface_weights/resolve/main"
SIZES = {"Alignment_RetinaFace.pth": 1789735, "MTL_backbone.pth": 101710914, "Landmark_98.pkl": 178030153}
for f, size in SIZES.items():
    dst = f"{OF}/weights/{f}"
    if not (os.path.exists(dst) and os.path.getsize(dst) == size):
        print("downloading", f); urllib.request.urlretrieve(f"{BASE}/{f}", dst)
    assert os.path.getsize(dst) == size, f"{f}: wrong size, delete it and rerun"
print("weights OK")

import importlib, importlib.util, pathlib
importlib.invalidate_caches()
p = pathlib.Path(importlib.util.find_spec("openface").submodule_search_locations[0]) / "STAR/conf/alignment.py"
s = p.read_text()
s2 = re.sub(r"self\.ckpt_dir = '/work/[^']*'", "self.ckpt_dir = osp.expanduser('~/openface_star_workdir')", s)
if s2 != s:
    p.write_text(s2)
print("patch:", [l.strip() for l in p.read_text().splitlines() if "ckpt_dir =" in l][0])

## 12. Smoke test on one clip
Expect: no errors, face found in ~100% of frames, ~100 rows (10 s at 10 fps), and an `.npz` with landmarks + embedding.

In [ ]:
import pandas as pd, numpy as np
one = sorted(glob.glob(f"{DATASET}/**/*.avi", recursive=True))[0]
!python src/extract_features.py "{one}" --out /content/smoke --device cuda --overwrite
stem = os.path.splitext(os.path.basename(one))[0]
df = pd.read_csv(f"/content/smoke/{stem}.csv")
z = np.load(f"/content/smoke/{stem}.npz")
print(df.shape, "| face found:", f"{df.face_detected.mean():.0%}")
print({k: (z[k].shape, z[k].dtype) for k in z.files})
df.head()

## 13. Extract everything to Drive (hours; keep the tab open)
Free Colab sessions end after several hours or when idle. Just rerun cells 1 → 5, 11, 13:
finished clips are skipped (`N already done` in the first line). A clip that fails is listed at the end, not fatal.

In [ ]:
!python src/extract_features.py "{DATASET}" --out {DRIVE_OUT}/features --device cuda --fps 10

## 14. Check completeness

In [ ]:
vids = glob.glob(f"{DATASET}/**/*.avi", recursive=True)
csvs = glob.glob(f"{DRIVE_OUT}/features/**/*.csv", recursive=True)
print(f"{len(csvs)} / {len(vids)} clips extracted")
!du -sh {DRIVE_OUT}/features